# 의료 도메인 BERT 프리트레인 말뭉치 생성
**국문 원천데이터(JSON) + 154 의료/법률 서적(TXT) → MLM+NSP 토크나이즈 완료 데이터셋**

In [1]:
import os, re, json, html, random, string, gc
from collections import Counter
from pathlib import Path
from datasets import Dataset, Features, Value, ClassLabel, load_from_disk
from transformers import AutoTokenizer
from tqdm import tqdm

In [2]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
PREP = os.path.join(REPO_ROOT, "data", "2.preprocessing")

# ==========================================
# [설정]
# ==========================================
BASE_DIR = os.path.join(REPO_ROOT, "data", "1.corpus", "2.aihub", "origin")

# 국문 원천데이터 JSON 디렉토리 (08, 09)
JSON_DIRS = [
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_기타',
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_온라인 의료 정보 제공 사이트',
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_의학 교과서',
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_학술 논문 및 저널',
    f'{BASE_DIR}/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_학회 가이드라인',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_기타',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_온라인 의료 정보 제공 사이트',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_의학 교과서',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_학술 논문 및 저널',
    f'{BASE_DIR}/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_학회 가이드라인',
]

# 154 의료/법률 서적 TXT (Training + Validation 모두 프리트레인에 사용)
TXT_DIRS = [
    f'{BASE_DIR}/154.의료, 법률 전문 서적 말뭉치/Training/01.원천데이터/TS_01.의료',
    f'{BASE_DIR}/154.의료, 법률 전문 서적 말뭉치/Validation/01.원천데이터/VS_01.의료'
]

INTERMEDIATE_DIR = f'{BASE_DIR}/output/intermediate_docs'
OUTPUT_DIR       = f'{BASE_DIR}/output/bert_pretrain_dataset'
MODEL_NAME       = 'klue/bert-base'
MAX_SEQ_LENGTH   = 512
TARGET_SEG_LEN   = 250
VAL_SIZE         = 5000
SEED             = 42

NUM_PROC = 42

random.seed(SEED)
os.makedirs(f'{BASE_DIR}/output', exist_ok=True)
print('설정 완료')


설정 완료


## 단계 0: 파일 탐색 및 샘플 확인

In [3]:
print('=' * 60)
json_files, txt_files = [], []
for d in JSON_DIRS:
    p = Path(d)
    if p.exists():
        fs = list(p.glob('*.json'))
        json_files.extend(fs)
        print(f'  JSON [{len(fs):>6}]  {p.name}')
    else:
        print(f'  [MISSING] {d}')
for d in TXT_DIRS:
    p = Path(d)
    if p.exists():
        fs = list(p.glob('*.txt'))
        txt_files.extend(fs)
        print(f'  TXT  [{len(fs):>6}]  {p.name}')
    else:
        print(f'  [MISSING] {d}')
print('-' * 60)
print(f'  총 JSON: {len(json_files):,}  |  총 TXT: {len(txt_files):,}  |  합계: {len(json_files)+len(txt_files):,}')


  [MISSING] ./datasets/aihub/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_기타
  [MISSING] ./datasets/aihub/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_온라인 의료 정보 제공 사이트
  [MISSING] ./datasets/aihub/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_의학 교과서
  [MISSING] ./datasets/aihub/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_학술 논문 및 저널
  [MISSING] ./datasets/aihub/08.전문 의학지식 데이터/Training/01.원천데이터/TS_국문_학회 가이드라인
  [MISSING] ./datasets/aihub/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_기타
  [MISSING] ./datasets/aihub/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_온라인 의료 정보 제공 사이트
  [MISSING] ./datasets/aihub/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_의학 교과서
  [MISSING] ./datasets/aihub/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_학술 논문 및 저널
  [MISSING] ./datasets/aihub/09.필수의료 의학지식 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_국문_학회 가이드라인
  [MISSING] ./datasets/aihub/154.의료, 법률 전문 서적 말뭉치/Training/01.원천데이터/TS_01.의료
  [MISSING] ./datasets/aihub/154.의료, 법률 전문 서적 말뭉치/Validation/01.원천데이터

In [4]:
print('[JSON 샘플]')
for fp in json_files[:2]:
    with open(fp, encoding='utf-8-sig') as f:
        d = json.load(f)
    content = d.get('content', '')
    print(f'  파일: {fp.name}  키: {list(d.keys())}')
    print(f'  content ({len(content)}자): {content[:120]}')
    print()
print('[TXT 샘플]')
if txt_files:
    with open(txt_files[0], encoding='utf-8-sig') as f:
        print(f'  파일: {txt_files[0].name}')
        print(f'  {f.read(200)}')


[JSON 샘플]
[TXT 샘플]


## 단계 1: 텍스트 정제 함수 정의

In [5]:
# ==========================================
# [단계 1] 텍스트 정제 (Hangeulize 없음 - 의료 영문 용어 보존)
# ==========================================
PUNCT_MAPPING = {
    '\u2018': "'", '\u2019': "'", '\u201c': '"', '\u201d': '"',
    '\u20b9': 'e', '\u00b4': "'", '\u00b0': '', '\u20ac': 'e', '\u2122': 'tm',
    '\u221a': ' sqrt ', '\u00d7': 'x', '\u00b2': '2', '\u2014': '-', '\u2013': '-',
    '\u00a3': 'e', '\u221e': 'infinity', '\u03b8': 'theta', '\u00f7': '/',
    '\u03b1': 'alpha', '\u2022': '.', '\u00e0': 'a', '\u2212': '-',
    '\u03b2': 'beta', '\u2205': '', '\u00b3': '3', '\u03c0': 'pi',
    '\u200b': ' ', '\u2026': ' ... ', '\ufeff': '', '\u00b7': '.'
}
_PUNCT     = re.escape(string.punctuation)
_WHITELIST = re.compile(rf'[^A-Za-z0-9\u3131-\u3163\uac00-\ud7a3{_PUNCT}\s]')
_HTML      = re.compile(r'<[^>]*>')
_SPACE     = re.compile(r'\s+')


def clean_str(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = _HTML.sub('', text)
    for p, r in PUNCT_MAPPING.items():
        text = text.replace(p, r)
    text = _WHITELIST.sub('', text)
    return _SPACE.sub(' ', text).strip()


def clean_text_medical(text: str) -> str:
    if not isinstance(text, str) or not text:
        return ''
    text = html.unescape(text)
    text = re.sub(r'(https?://|www\.)[A-Za-z0-9\uac00-\ud7a3./?=&%~_#-]+', '', text)
    text = re.sub(r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}', '', text)
    text = re.sub(r' ?\(\s*\) ?', ' ', text)
    text = re.sub(r'\(\.+\)', '', text)
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r'외부\s*링크[:.]*', '', text)
    text = re.sub(r'Copyright.*', '', text, flags=re.IGNORECASE)
    return clean_str(text)


def is_spam(text: str) -> bool:
    length = len(text)
    if length < 20:
        return True
    boilerplate = ['저작권', '크리에이티브 커먼즈', '라이선스', '분류:', '목차',
                   '파일:', '각주', '편집', '토론', '역사']
    if length < 50 and any(k in text for k in boilerplate):
        return True
    if text.strip().startswith(('*', '>', '-')) and len(text.strip()[1:].strip()) < 5:
        return True
    if length < 300:
        return False
    ns = text.replace(' ', '')
    if not ns:
        return True
    valid = len(re.sub(r'[^\uac00-\ud7a3a-zA-Z]', '', ns))
    if valid / len(ns) < 0.20:
        return True
    words = text.split()
    if words:
        _, cnt = Counter(words).most_common(1)[0]
        if cnt / len(words) > 0.6:
            return True
    if length > 1000 and text.count(' ') / length < 0.01:
        return True
    return False


def sentence_split(text: str):
    sents = re.split(r'(?<=[.?!])\s+', text)
    return [s.strip() for s in sents if len(s.strip()) > 5]


# 빠른 테스트
sample = '환자는 HTN, DM2 진단받았으며 ACE inhibitor 투여 중. BP 140/90mmHg, HbA1c 8.2% 확인됨.'
print('정제 테스트:', clean_text_medical(sample))


정제 테스트: 환자는 HTN, DM2 진단받았으며 ACE inhibitor 투여 중. BP 140/90mmHg, HbA1c 8.2% 확인됨.


## 단계 2: 문서 인덱스 생성 (Phase 1)

In [6]:
def doc_generator():
    stats = Counter()

    # JSON 파일: content 필드 추출
    for d in JSON_DIRS:
        p = Path(d)
        if not p.exists():
            continue
        for fpath in p.glob('*.json'):
            try:
                with open(fpath, encoding='utf-8-sig') as f:
                    data = json.load(f)
                raw = data.get('content', '')
                if not raw:
                    stats['json_skip'] += 1; continue
                cleaned = clean_text_medical(raw)
                if not cleaned or is_spam(cleaned):
                    stats['json_skip'] += 1; continue
                sents = sentence_split(cleaned)
                if sents:
                    stats['json_ok'] += 1
                    yield {'sentences': sents}
                else:
                    stats['json_skip'] += 1
            except Exception:
                stats['json_err'] += 1

    # TXT 파일: 빈 줄 기준 문단 분리 -> 각 문단을 하나의 문서로
    for d in TXT_DIRS:
        p = Path(d)
        if not p.exists():
            continue
        for fpath in p.glob('*.txt'):
            try:
                try:
                    raw = fpath.read_text(encoding='utf-8')
                except UnicodeDecodeError:
                    raw = fpath.read_text(encoding='cp949')
                paragraphs = [pr.strip() for pr in re.split(r'\n{2,}', raw) if pr.strip()]
                for para in paragraphs:
                    cleaned = clean_text_medical(para)
                    if not cleaned or is_spam(cleaned):
                        continue
                    sents = sentence_split(cleaned)
                    if sents:
                        stats['txt_ok'] += 1
                        yield {'sentences': sents}
            except Exception:
                stats['txt_err'] += 1

    print(f'  JSON 사용:{stats["json_ok"]:,} 제외:{stats["json_skip"]:,} 에러:{stats["json_err"]:,}')
    print(f'  TXT  사용:{stats["txt_ok"]:,}  에러:{stats["txt_err"]:,}')

print('doc_generator 정의 완료')

doc_generator 정의 완료


In [7]:
if not os.path.exists(INTERMEDIATE_DIR):
    print('[Phase 1] 문서 인덱스 생성 중...')
    doc_dataset = Dataset.from_generator(
        doc_generator,
        features=Features({'sentences': [Value('string')]}),
        writer_batch_size=500
    )
    doc_dataset.save_to_disk(INTERMEDIATE_DIR)
    print(f'[Phase 1] 완료: {len(doc_dataset):,} 문서')
else:
    print(f'[Phase 1] 기존 인덱스 로드: {INTERMEDIATE_DIR}')
    doc_dataset = load_from_disk(INTERMEDIATE_DIR)

print(f'총 문서 수: {len(doc_dataset):,}')


[Phase 1] 문서 인덱스 생성 중...


Generating train split: 0 examples [00:00, ? examples/s]

  JSON 사용:0 제외:0 에러:0
  TXT  사용:0  에러:0


ValueError: Instruction "train" corresponds to no data!

## 단계 3: 통계 확인

In [ ]:
import statistics as stat_mod

total_sents, total_chars = 0, 0
sampled_lens = []

for item in tqdm(doc_dataset, ncols=80, desc='통계'):
    sents = item['sentences']
    total_sents += len(sents)
    for s in sents:
        total_chars += len(s)
    if len(sampled_lens) < 20000:
        sampled_lens.extend(len(s) for s in sents)

print('\n' + '='*50)
print(f'  총 문서 수     : {len(doc_dataset):>10,}')
print(f'  총 문장 수     : {total_sents:>10,}')
print(f'  총 글자 수     : {total_chars:>10,}')
if total_sents:
    print(f'  평균 문장 길이 : {total_chars/total_sents:.1f} 자')
if sampled_lens:
    print(f'  문장 중앙값    : {stat_mod.median(sampled_lens):.0f} 자 (샘플 {len(sampled_lens):,}개)')
print('='*50)


## 단계 4: NSP 쌍 생성 (Phase 2)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'토크나이저: {MODEL_NAME}  어휘:{tokenizer.vocab_size:,}')


def create_segments(sentences, tokenizer, max_len, target_len):
    """문장 리스트를 target_len 토큰 단위 세그먼트로 패킹"""
    segments, cur_sents, cur_len = [], [], 0
    for sent in sentences:
        ids  = tokenizer.encode(sent, add_special_tokens=False)
        slen = len(ids)
        if slen > max_len - 20:
            if cur_sents:
                segments.append(' '.join(cur_sents))
                cur_sents, cur_len = [], 0
            for i in range(0, slen, target_len):
                segments.append(tokenizer.decode(ids[i:i+target_len]))
            continue
        if cur_len + slen < target_len:
            cur_sents.append(sent); cur_len += slen
        else:
            if cur_sents:
                segments.append(' '.join(cur_sents))
            cur_sents, cur_len = [sent], slen
    if cur_sents:
        segments.append(' '.join(cur_sents))
    return segments


def nsp_gen_wrapper(doc_dataset, tokenizer, max_len, target_len):
    """NSP 쌍 생성: 50% IsNext / 50% NotNext"""
    total  = len(doc_dataset)
    offset = total // 2
    for doc_idx, item in enumerate(tqdm(doc_dataset, desc='NSP 생성', ncols=80, mininterval=10)):
        if doc_idx % 5000 == 0:
            gc.collect()
        sents = item['sentences']
        if not sents:
            continue
        segs = create_segments(sents, tokenizer, max_len, target_len)
        if len(segs) < 2:
            continue
        for i in range(len(segs) - 1):
            seg_a = segs[i]
            if random.random() < 0.5:
                seg_b = segs[i + 1]; label = 0  # IsNext
            else:
                rand_idx   = (doc_idx + offset + random.randint(1, max(1, total // 4))) % total
                rand_sents = doc_dataset[rand_idx]['sentences']
                if rand_sents:
                    start = random.randint(0, max(0, len(rand_sents) - 5))
                    seg_b = ' '.join(rand_sents[start:start + 5]); label = 1  # NotNext
                else:
                    seg_b = segs[i + 1]; label = 0
            yield {'sentence_a': seg_a, 'sentence_b': seg_b, 'next_sentence_label': label}


print('NSP 함수 정의 완료')


In [ ]:
print('[Phase 2] NSP 쌍 생성 중...')

nsp_features = Features({
    'sentence_a': Value('string'),
    'sentence_b': Value('string'),
    'next_sentence_label': ClassLabel(num_classes=2, names=['IsNext', 'NotNext'])
})

def nsp_gen():
    yield from nsp_gen_wrapper(doc_dataset, tokenizer, MAX_SEQ_LENGTH, TARGET_SEG_LEN)

nsp_dataset = Dataset.from_generator(nsp_gen, features=nsp_features, writer_batch_size=200)

labels   = nsp_dataset['next_sentence_label']
is_next  = sum(1 for l in labels if l == 0)
not_next = len(labels) - is_next
print(f'총 NSP 쌍: {len(nsp_dataset):,}')
print(f'  IsNext  (0): {is_next:,} ({100*is_next/len(labels):.1f}%)')
print(f'  NotNext (1): {not_next:,} ({100*not_next/len(labels):.1f}%)')


## 단계 5: 토크나이징 및 저장

In [ ]:
print(f'[분리] train/val (val={VAL_SIZE:,})...')
split = nsp_dataset.train_test_split(
    test_size=VAL_SIZE, seed=SEED, shuffle=True, keep_in_memory=False
)
print(f'  train: {len(split["train"]):,}  |  val: {len(split["test"]):,}')


def tokenize_fn(examples):
    return tokenizer(
        examples['sentence_a'],
        examples['sentence_b'],
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding='max_length',
        return_token_type_ids=True,
        return_special_tokens_mask=True,
    )


print('[토크나이징]...')
tokenized = split.map(
    tokenize_fn,
    batched=True,
    batch_size=512,
    num_proc=NUM_PROC,
    remove_columns=['sentence_a', 'sentence_b'],
    desc='토크나이징'
)

print(f'[저장] {OUTPUT_DIR}')
tokenized.save_to_disk(OUTPUT_DIR)
print('완료!')
print(f'  컬럼: {tokenized["train"].column_names}')


## 단계 6: 저장 검증

In [ ]:
ds = load_from_disk(OUTPUT_DIR)
s  = ds['train'][0]
print('컬럼:', list(s.keys()))
print('input_ids 길이:', len(s['input_ids']))
print('token_type_ids (앞30):', s['token_type_ids'][:30])
print('special_tokens_mask (앞30):', s['special_tokens_mask'][:30])
print('next_sentence_label:', s['next_sentence_label'])
print('\n디코딩 (첫 60토큰):')
print(tokenizer.decode(s['input_ids'][:60]))
